In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

データ確認

In [ ]:
import pandas as pd
train = pd.read_csv('/kaggle/input/nlp-getting-started/test.csv')
train.head()

In [ ]:
train.columns

災害ツイートの割合について確認

In [ ]:
train.describe()

欠損値を見る

In [ ]:
train.isnull().sum()



以上より，ツイートの約43%が災害ツイートであることが確認できた
idについては関係ないため除外

In [ ]:
train.drop(['id'], axis=1, inplace=True)

*****keywordの欠損値を’Nokeyword’に置き換える

In [ ]:
train['keyword_filled'] = train['keyword'].fillna('NoKeyword')

In [ ]:
if 'target' in train.columns and 'keyword_filled' in train.columns:
    keyword_target_rate = train.groupby('keyword_filled')['target'].mean()
    print(keyword_target_rate)
else:
    print("必要なカラム（'keyword_filled' または 'target'）がtrainに存在しません。")
    keyword_target_rate = None  # エラー回避用にNoneを代入

keywordごとの災害ツイート率

In [ ]:
print(keyword_target_rate)

keywordは特徴量として使用できそう

locationごとの災害ツイート率

In [ ]:
train['location_filled'] = train['location'].fillna('NoLocation')

In [ ]:
if 'target' in train.columns and 'location_filled' in train.columns:
    location_summary = (
        train
          .groupby('location_filled')['target']
          .agg(disaster_rate='mean', count='size')
          .sort_values(by='count', ascending=False)
    )
    print(location_summary.head(20))
else:
    print("必要なカラム（'location_filled' または 'target'）がtrainに存在しません。")
    location_summary = None  # 後続エラー回避用

locationは関係なさそう

textの文字数と災害ツイートの関係

In [ ]:
train['text_length'] = train['text'].str.len()
print(train['text_length'].describe())

In [ ]:
if 'target' in train.columns and 'text_length' in train.columns:
    print(train.groupby('target')['text_length'].mean())
else:
    print("必要なカラム（'target' または 'text_length'）がtrainに存在しません。")

災害ツイートの方が14%ほど文字数が多い傾向

textの単語数と災害ツイートの関係

In [ ]:
train['word_count'] = train['text'].str.split().apply(len)
print(train['word_count'].describe())

In [ ]:
if 'target' in train.columns and 'word_count' in train.columns:
    print(train.groupby('target')['word_count'].mean())
else:
    print("必要なカラム（'target' または 'word_count'）がtrainに存在しません。")

単語数の平均値にはほとんど差がなく，ツイートの内容とは関係なさそう

keywordと文字数を特徴量として，ロジスティック回帰によりモデル学習

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# データ読み込み
train = pd.read_csv('/kaggle/input/nlp-getting-started/train.csv')
test = pd.read_csv('/kaggle/input/nlp-getting-started/test.csv')

# 欠損補完
train['keyword_filled'] = train['keyword'].fillna('NoKeyword')
test['keyword_filled'] = test['keyword'].fillna('NoKeyword')

# keyword をワンホットエンコーディング
encoder = OneHotEncoder(handle_unknown='ignore', sparse=False)
keyword_train_encoded = encoder.fit_transform(train[['keyword_filled']])
keyword_test_encoded = encoder.transform(test[['keyword_filled']])

# text の文字数を特徴量に追加
train['text_length'] = train['text'].str.len()
test['text_length'] = test['text'].str.len()

# keyword + 文字数 を特徴量として結合
X_train = np.hstack([keyword_train_encoded, train[['text_length']].values])
X_test = np.hstack([keyword_test_encoded, test[['text_length']].values])
y_train = train['target']

# ← ここまでが重要：X_train を作ってから使う！

# 学習・検証に分割（これで NameError は出ない）
X_tr, X_val, y_tr, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)

# モデル学習
model = LogisticRegression(max_iter=200)
model.fit(X_tr, y_tr)

# 精度評価
val_preds = model.predict(X_val)
print("Accuracy:", accuracy_score(y_val, val_preds))
print("Confusion Matrix:\n", confusion_matrix(y_val, val_preds))
print("Classification Report:\n", classification_report(y_val, val_preds))



モデルの精度検証

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
val_preds = model.predict(X_val)
print("Accuracy:", accuracy_score(y_val, val_preds))
print("Confusion Matrix:\n", confusion_matrix(y_val, val_preds))
print("Classification Report:\n", classification_report(y_val, val_preds))

約73％の精度で予測できていることが確認できたが，改善の余地あり

テキストの内容について検討するため単語の出現頻度を確認

In [ ]:
import pandas as pd
from collections import Counter
import re

# データ読み込み
train = pd.read_csv('/kaggle/input/nlp-getting-started/train.csv')

# 欠損値を空文字にしておく
train['text'] = train['text'].fillna('')

# 小文字化 & 記号除去
def clean(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)  # アルファベットと空白以外を除去
    return text

# 全ツイートの単語を一つのリストにまとめる
all_words = []
for sentence in train['text']:
    cleaned = clean(sentence)
    words = cleaned.split()
    all_words.extend(words)

# 単語ごとの出現回数をカウント
word_freq = Counter(all_words)

# 上位20単語を表示
print("単語出現頻度（上位20語）")
for word, freq in word_freq.most_common(20):
    print(f"{word:15s} : {freq}")

In [ ]:
import pandas as pd
from collections import Counter
import re

# データ読み込み
train = pd.read_csv('/kaggle/input/nlp-getting-started/train.csv')

# 欠損値を空文字にしておく
train['text'] = train['text'].fillna('')

# 小文字化 & 記号除去 
def clean(text):
    text = text.lower() 
    text = re.sub(r'[^a-z\s]', '', text)  # アルファベットと空白以外を除去
    return text

# 全ツイートの単語を一つのリストにまとめる
all_words = []
for sentence in train['text']:
    cleaned = clean(sentence)
    words = cleaned.split()
    all_words.extend(words)

# 単語ごとの出現回数をカウント
word_freq = Counter(all_words)

# 上位20単語を表示
print("単語出現頻度（上位20語）")
for word, freq in word_freq.most_common(20):
    print(f"{word:15s} : {freq}")

In [ ]:
import re
from collections import Counter
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords

train['text'] = train['text'].fillna('')

# 前処理関数（小文字化＋記号除去）
def clean(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    return text

# ストップワード除去用セット
stop_words = set(stopwords.words('english'))

# 災害ツイート (target=1)
disaster_words = []
for text in train[train['target'] == 1]['text']:
    words = clean(text).split()
    words = [w for w in words if w not in stop_words]
    disaster_words.extend(words)

# 非災害ツイート (target=0)
nondisaster_words = []
for text in train[train['target'] == 0]['text']:
    words = clean(text).split()
    words = [w for w in words if w not in stop_words]
    nondisaster_words.extend(words)

import re
from collections import Counter
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords

train['text'] = train['text'].fillna('')

# 前処理関数（小文字化＋記号除去）
def clean(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    return text

# ストップワード除去用セット
stop_words = set(stopwords.words('english'))

# 災害ツイート (target=1)
disaster_words = []
for text in train[train['target'] == 1]['text']:
    words = clean(text).split()
    words = [w for w in words if w not in stop_words]
    disaster_words.extend(words)

# 非災害ツイート (target=0)
nondisaster_words = []
for text in train[train['target'] == 0]['text']:
    words = clean(text).split()
    words = [w for w in words if w not in stop_words]
    nondisaster_words.extend(words)

In [ ]:
# 必要なライブラリをインポート
import pandas as pd
import re
import nltk
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# nltkのストップワードをダウンロード
nltk.download('stopwords')
from nltk.corpus import stopwords

# ① モデルの構築 ----------------------------------------

# データ読み込み
train = pd.read_csv('/kaggle/input/nlp-getting-started/train.csv')
test = pd.read_csv('/kaggle/input/nlp-getting-started/test.csv')

# 欠損値を空文字で補完
train['text'] = train['text'].fillna('')
test['text'] = test['text'].fillna('')

# 前処理関数：小文字化 & 記号除去
def clean(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    return text

# テキストを前処理
train['text_clean'] = train['text'].apply(clean)
test['text_clean'] = test['text'].apply(clean)

# TF-IDFベクトル化（英語のストップワードを除去）
stop_words = stopwords.words('english')
vectorizer = TfidfVectorizer(max_features=5000, stop_words=stop_words)
X = vectorizer.fit_transform(train['text_clean'])
X_test = vectorizer.transform(test['text_clean'])
y = train['target']

# ② モデル精度の検証 ----------------------------------------

# 学習用と検証用に分割（80%:20%）
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# ロジスティック回帰モデルの学習
model = LogisticRegression(max_iter=200)
model.fit(X_train, y_train)

# 検証データで予測
val_preds = model.predict(X_val)

# 精度の表示
print("モデル精度（バリデーションデータ）")
print("Accuracy:", accuracy_score(y_val, val_preds))
print("\nConfusion Matrix:\n", confusion_matrix(y_val, val_preds))
print("\nClassification Report:\n", classification_report(y_val, val_preds))

# ③ テストデータの予測と提出 ----------------------------------------

# test.csv の予測
test_preds = model.predict(X_test)

# 提出用CSVファイル作成
submission = pd.DataFrame({
    'id': test['id'],
    'target': test_preds
})
submission.to_csv('submission.csv', index=False)